In [2]:
"""
MMD^2 and KL Divergence for ReVeal (Debian -> Chrome)
------------------------------------------------------------
"""

import os
import gc
import json
import time
import traceback
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import rbf_kernel
from imblearn.over_sampling import SMOTE, ADASYN, SVMSMOTE
from scipy.stats import entropy
import warnings

warnings.filterwarnings('ignore')


DATASET_DIR  = "../../../../../embedding/reveal/codebert/"
RESULTS_PATH = "results/mmd_kl_reveal.json"
SEED         = 42

PCA_COMPONENTS = 50
KL_BINS        = 50
MMD_GAMMA      = 1.0
MAX_SAMPLES    = 20000

GAN_LATENT_DIM  = 100
GAN_EPOCHS      = 1500
GAN_BATCH_SIZE  = 64
GAN_LOG_EVERY   = 500

os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"

METHODS = ["No Oversampling", "SMOTE", "ADASYN", "SVMSMOTE", "GAN"]

CACHE_FILES = {
    "SMOTE": f"{DATASET_DIR}/debian_smote.npy",
    "ADASYN": f"{DATASET_DIR}/debian_adasyn.npy",
    "SVMSMOTE": f"{DATASET_DIR}/debian_svmsmote.npy",
    "GAN": f"{DATASET_DIR}/debian_gan_embeddings.npy",
}


class Generator(nn.Module):
    def __init__(self, latent_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


def load_results():
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH, "r") as f:
            return json.load(f)
    return {}


def save_results(results):
    tmp_path = RESULTS_PATH + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(results, f, indent=2)
    os.replace(tmp_path, RESULTS_PATH)


def print_progress_summary(results):
    done = sum(1 for v in results.values() if v.get("status") == "ok")
    failed = sum(1 for v in results.values() if v.get("status") == "failed")
    remaining = len(METHODS) - done - failed
    pct = 100 * (done + failed) / len(METHODS)
    print(f"----- PROGRESS: {done} ok | {failed} failed | {remaining} remaining "
          f"| {pct:.1f}% of {len(METHODS)} methods -----")


def load_data():
    print("Loading embeddings...")
    deb_emb = np.load(f"{DATASET_DIR}/debian_embeddings.npy").astype(np.float32)
    deb_lbl = np.load(f"{DATASET_DIR}/debian_labels.npy").astype(np.float32)
    chr_emb = np.load(f"{DATASET_DIR}/chrome_embeddings.npy").astype(np.float32)
    chr_lbl = np.load(f"{DATASET_DIR}/chrome_labels.npy").astype(np.float32)

    scaler = StandardScaler()
    deb_emb_scaled = scaler.fit_transform(deb_emb).astype(np.float32)
    chr_emb_scaled = scaler.transform(chr_emb).astype(np.float32)

    print(f"Debian (source): {deb_emb_scaled.shape} (vuln={int(deb_lbl.sum())}) | "
          f"Chrome (target): {chr_emb_scaled.shape} (vuln={int(chr_lbl.sum())})")

    return deb_emb_scaled, deb_lbl, chr_emb_scaled, chr_lbl


def generate_gan_embeddings(source_emb, source_lbl):
    input_dim = source_emb.shape[1]
    minority_emb = source_emb[source_lbl == 1].astype(np.float32)
    n_to_generate = int(np.sum(source_lbl == 0)) - int(np.sum(source_lbl == 1))

    print(f"    Minority samples: {minority_emb.shape[0]} | Target to generate: {n_to_generate}")

    G = Generator(GAN_LATENT_DIM, input_dim).to(DEVICE)
    D = Discriminator(input_dim).to(DEVICE)
    opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
    criterion = nn.BCELoss()

    minority_tensor = torch.tensor(minority_emb).to(DEVICE)
    dataset = TensorDataset(minority_tensor)
    loader = DataLoader(dataset, batch_size=GAN_BATCH_SIZE, shuffle=True)

    t0 = time.time()
    epoch_bar = tqdm(range(GAN_EPOCHS), desc="    GAN training", leave=False)
    for epoch in epoch_bar:
        for (real_batch,) in loader:
            bs = real_batch.size(0)

            z = torch.randn(bs, GAN_LATENT_DIM).to(DEVICE)
            fake = G(z).detach()
            loss_D = criterion(D(real_batch), torch.ones(bs, 1).to(DEVICE)) + \
                     criterion(D(fake), torch.zeros(bs, 1).to(DEVICE))
            opt_D.zero_grad()
            loss_D.backward()
            opt_D.step()

            z = torch.randn(bs, GAN_LATENT_DIM).to(DEVICE)
            loss_G = criterion(D(G(z)), torch.ones(bs, 1).to(DEVICE))
            opt_G.zero_grad()
            loss_G.backward()
            opt_G.step()

        if (epoch + 1) % GAN_LOG_EVERY == 0:
            epoch_bar.set_postfix({"D_loss": f"{loss_D.item():.4f}", "G_loss": f"{loss_G.item():.4f}"})

    print(f"    GAN training complete in {time.time() - t0:.1f}s")

    G.eval()
    with torch.no_grad():
        z = torch.randn(n_to_generate, GAN_LATENT_DIM).to(DEVICE)
        synthetic = G(z).cpu().numpy()

    resampled_emb = np.vstack([source_emb, synthetic]).astype(np.float32)

    del G, D, opt_G, opt_D, minority_tensor, dataset, loader
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    elif DEVICE == "cuda":
        torch.cuda.empty_cache()

    return resampled_emb


def load_source_for_method(method, deb_emb_scaled, deb_lbl):
    if method == "No Oversampling":
        return deb_emb_scaled

    cache_path = CACHE_FILES[method]
    if os.path.exists(cache_path):
        print(f"    Loading cached {method} embeddings from disk...")
        return np.load(cache_path).astype(np.float32)

    print(f"    No cache found for {method} -- generating now...")
    t0 = time.time()

    if method == "SMOTE":
        sampler = SMOTE(random_state=SEED)
        resampled_emb, _ = sampler.fit_resample(deb_emb_scaled, deb_lbl)
    elif method == "ADASYN":
        sampler = ADASYN(random_state=SEED)
        resampled_emb, _ = sampler.fit_resample(deb_emb_scaled, deb_lbl)
    elif method == "SVMSMOTE":
        sampler = SVMSMOTE(random_state=SEED)
        resampled_emb, _ = sampler.fit_resample(deb_emb_scaled, deb_lbl)
    elif method == "GAN":
        resampled_emb = generate_gan_embeddings(deb_emb_scaled, deb_lbl)
    else:
        raise ValueError(f"Unknown method: {method}")

    resampled_emb = resampled_emb.astype(np.float32)
    np.save(cache_path, resampled_emb)
    print(f"    Generated and cached {method} embeddings in {time.time() - t0:.1f}s. "
          f"Shape: {deb_emb_scaled.shape} -> {resampled_emb.shape}")

    return resampled_emb


def compute_mmd(X, Y, gamma=MMD_GAMMA):
    XX = rbf_kernel(X, X, gamma=gamma)
    YY = rbf_kernel(Y, Y, gamma=gamma)
    XY = rbf_kernel(X, Y, gamma=gamma)
    return float(XX.mean() + YY.mean() - 2 * XY.mean())


def compute_avg_kl(source_pca, target_pca, bins=KL_BINS):
    kl_values = []
    for i in range(source_pca.shape[1]):
        hist_source, edges = np.histogram(source_pca[:, i], bins=bins, density=True)
        hist_target, _ = np.histogram(target_pca[:, i], bins=edges, density=True)

        hist_source = hist_source + 1e-10
        hist_target = hist_target + 1e-10

        kl = entropy(hist_target, hist_source)
        kl_values.append(kl)

    return float(np.mean(kl_values))


def run_method(method, deb_emb_scaled, deb_lbl, chr_emb_scaled):
    source_for_mmd = load_source_for_method(method, deb_emb_scaled, deb_lbl)

    n_source = min(source_for_mmd.shape[0], MAX_SAMPLES)
    n_target = min(chr_emb_scaled.shape[0], MAX_SAMPLES)

    rng = np.random.RandomState(SEED)
    source_sample = source_for_mmd[rng.choice(source_for_mmd.shape[0], n_source, replace=False)] \
        if source_for_mmd.shape[0] > n_source else source_for_mmd
    target_sample = chr_emb_scaled[rng.choice(chr_emb_scaled.shape[0], n_target, replace=False)] \
        if chr_emb_scaled.shape[0] > n_target else chr_emb_scaled

    pca = PCA(n_components=PCA_COMPONENTS, random_state=SEED)
    pca.fit(np.vstack([source_sample, target_sample]))

    source_pca = pca.transform(source_sample)
    target_pca = pca.transform(target_sample)

    mmd = compute_mmd(source_pca, target_pca)
    avg_kl = compute_avg_kl(source_pca, target_pca)

    return {
        "mmd_squared": round(mmd, 6),
        "avg_kl_divergence": round(avg_kl, 6),
        "n_source_used": int(source_sample.shape[0]),
        "n_target_used": int(target_sample.shape[0]),
        "n_source_full": int(source_for_mmd.shape[0]),
        "n_target_full": int(chr_emb_scaled.shape[0]),
    }


def main():
    results = load_results()
    print(f"Loaded {len(results)} existing results from {RESULTS_PATH}")
    print_progress_summary(results)

    deb_emb_scaled, deb_lbl, chr_emb_scaled, chr_lbl = load_data()

    n_methods = len(METHODS)
    for m_idx, method in enumerate(METHODS, start=1):
        if method in results and results[method].get("status") == "ok":
            print(f"[{m_idx}/{n_methods}] {method}: already computed "
                  f"(MMD^2={results[method].get('mmd_squared', 'N/A')}) -- skipping.")
            continue

        print(f"\n[{m_idx}/{n_methods}] {method}:")
        t0 = time.time()

        try:
            metrics = run_method(method, deb_emb_scaled, deb_lbl, chr_emb_scaled)
            metrics["method"] = method
            metrics["elapsed_sec"] = round(time.time() - t0, 1)
            metrics["status"] = "ok"
            results[method] = metrics
            save_results(results)
            print(f"  MMD^2={metrics['mmd_squared']:.6f} | "
                  f"Avg KL={metrics['avg_kl_divergence']:.6f} "
                  f"({metrics['elapsed_sec']}s) -- saved.")
            print_progress_summary(results)

        except Exception as e:
            print(f"  FAILED: {e}")
            traceback.print_exc()
            results[method] = {
                "method": method,
                "status": "failed",
                "error": str(e),
                "elapsed_sec": round(time.time() - t0, 1),
            }
            save_results(results)
            print_progress_summary(results)

        finally:
            gc.collect()

    print("\nALL METHODS PROCESSED (OR ALREADY CACHED).")
    print_progress_summary(results)
    print(f"Final results saved to {RESULTS_PATH}")


if __name__ == "__main__":
    main()

Loaded 0 existing results from results/mmd_kl_reveal.json
----- PROGRESS: 0 ok | 0 failed | 5 remaining | 0.0% of 5 methods -----
Loading embeddings...
Debian (source): (18298, 768) (vuln=1415) | Chrome (target): (4436, 768) (vuln=825)

[1/5] No Oversampling:
  MMD^2=0.000384 | Avg KL=0.046079 (2.6s) -- saved.
----- PROGRESS: 1 ok | 0 failed | 4 remaining | 20.0% of 5 methods -----

[2/5] SMOTE:
    Loading cached SMOTE embeddings from disk...
  MMD^2=0.000510 | Avg KL=0.077778 (4.0s) -- saved.
----- PROGRESS: 2 ok | 0 failed | 3 remaining | 40.0% of 5 methods -----

[3/5] ADASYN:
    Loading cached ADASYN embeddings from disk...
  MMD^2=0.000496 | Avg KL=0.073848 (3.1s) -- saved.
----- PROGRESS: 3 ok | 0 failed | 2 remaining | 60.0% of 5 methods -----

[4/5] SVMSMOTE:
    Loading cached SVMSMOTE embeddings from disk...
  MMD^2=0.002188 | Avg KL=0.096173 (5.4s) -- saved.
----- PROGRESS: 4 ok | 0 failed | 1 remaining | 80.0% of 5 methods -----

[5/5] GAN:
    Loading cached GAN embeddin